# Sanity check: hybrid sampling (`reuse_frac`) in `optimize_LGD`

Runs the **2D cond 1D** setup, for both the plain diffusion LGD path and the Consistency-Model LGD-F path, once with `reuse_frac=0.0` (current behavior, unchanged) and once with `reuse_frac=0.5` (half of the `nsamples` MMD-comparison batch at each step is carried over from the previous step instead of freshly sampled) — and compares final MMD, wall time, and the per-step gradient-norm trajectory, to confirm the reuse path is actually wired up and behaving as expected before running the full sweep.

In [ ]:
import os
# ============================================================
# CONFIG — same as Exp_2D_cond_1D.ipynb (fastest setup, good for a quick sanity check)
# ============================================================
EXPERIMENT_NAME   = "2D_cond_1D"
GLOBAL_SEED       = 42
FORCE_RETRAIN     = False

BASE_DIR = os.path.normpath(os.path.join(os.getcwd(), ".."))
PARAMS_DIR        = f"{BASE_DIR}/params"
CHECKPOINT_DIR    = f"{BASE_DIR}/checkpoints/{EXPERIMENT_NAME}"
RESULTS_DIR       = f"{BASE_DIR}/results/{EXPERIMENT_NAME}"

# Architecture — Diffusion models
NBLOCKS           = 3
NUNITS            = 128

# Architecture — Consistency Model
NBLOCKS_CM        = 3
NUNITS_CM         = 128

# Training (only used if a checkpoint is missing)
NEPOCHS           = 20_000
BATCH_SIZE        = 1_024
NEPOCHS_CM        = 20_000
BATCH_SIZE_CM     = 1_024

# Diffusion
DIFFUSION_STEPS   = 100

# Optimization — hybrid-sampling sanity check
N_ATTEMP_OPTIM              = 10     # small: this is a sanity check, not the full sweep
NSAMPLES_IN_OPTIM_FOR_MMD   = 250
NUM_X_T_LGD                 = 3
NUM_X_T_LGD_CM              = 3
REUSE_FRACS                 = [0.0, 0.5]

# GMM dimensions
CONDITION_ON      = 1   # dim(x)=1, dim(y)=1

In [ ]:
import os, sys

# ── point Python at simulations/src where all .py modules live ──
src_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "src")
src_path = os.path.normpath(src_path)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"src path on sys.path: {src_path}")

In [ ]:
import os, sys, time, json
import importlib
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
from tqdm import trange

import Diffusion
import LossFunctions
import ConsistencyModels
import dist_utils
import Optimization
import experiment_utils
from ConsistencyModels import ConsistencyModeliCT
from LossFunctions import MMDLoss, RBF

for mod in [Diffusion, LossFunctions, ConsistencyModels,
            dist_utils, Optimization, experiment_utils]:
    importlib.reload(mod)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)
os.makedirs(PARAMS_DIR,     exist_ok=True)
print("Imports done.")

In [ ]:
env_info = experiment_utils.get_environment_info()
experiment_utils.print_environment_info(env_info)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## GMM Parameters

In [ ]:
# ============================================================
# GMM PARAMETERS
# Priority:
#   1. Load from PARAMS_DIR  (shared across runs, committed to repo)
#   2. Load from RESULTS_DIR (fallback from a previous run)
#   3. Generate fresh and save to both dirs
# ============================================================
FORCE_REGENERATE_PARAMS = False

def _load_params():
    loaded = experiment_utils.load_gmm_params(PARAMS_DIR, EXPERIMENT_NAME)
    if loaded is not None:
        print(f"[GMM] Loaded from PARAMS_DIR: {PARAMS_DIR}")
        return loaded
    loaded = experiment_utils.load_gmm_params(RESULTS_DIR, EXPERIMENT_NAME)
    if loaded is not None:
        print(f"[GMM] Loaded from RESULTS_DIR: {RESULTS_DIR}")
        return loaded
    return None

loaded = None if FORCE_REGENERATE_PARAMS else _load_params()

if loaded is not None:
    mu_list, Sigma_list, alpha, mog_means, mog_variances, weights, x_star = loaded
    mu_list    = [mu.float() for mu in mu_list]
    Sigma_list = [cov.float() for cov in Sigma_list]
    alpha      = alpha.float()
else:
    print("[GMM] Generating fresh parameters...")
    experiment_utils.set_global_seed(GLOBAL_SEED)
    mu_list = [
        torch.tensor([-5,  5], dtype=torch.float64),
        torch.tensor([-5, -5], dtype=torch.float64),
        torch.tensor([ 5,  3], dtype=torch.float64),
        torch.tensor([ 5, -1], dtype=torch.float64),
        torch.tensor([ 0, -3], dtype=torch.float64),
        torch.tensor([-2,  4], dtype=torch.float64),
        torch.tensor([-2, -3], dtype=torch.float64),
        torch.tensor([ 1,  2], dtype=torch.float64),
        torch.tensor([-8,  1], dtype=torch.float64),
        torch.tensor([ 7,  5], dtype=torch.float64),
        torch.tensor([ 0, -5], dtype=torch.float64),
    ]
    Sigma_list = [
        torch.tensor([[0.5000, 0.1950],
                      [0.1950, 0.2000]], dtype=torch.float64)
    ] * len(mu_list)
    alpha = torch.tensor([1 / len(mu_list)] * len(mu_list), dtype=torch.float64)

    mu_list    = [mu.float() for mu in mu_list]
    Sigma_list = [cov.float() for cov in Sigma_list]
    alpha      = alpha.float()

    x_star = torch.tensor([-5])
    mu_temp, Sigma_temp = dist_utils.compute_conditionals(mu_list, Sigma_list, x_star)
    temp_alpha          = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_star)
    mog_means, mog_variances, weights = dist_utils.filter_and_normalize(
        mu_temp, Sigma_temp, temp_alpha, threshold=0.01
    )

    experiment_utils.save_gmm_params(
        mu_list, Sigma_list, alpha,
        mog_means, mog_variances, weights, x_star,
        PARAMS_DIR, EXPERIMENT_NAME
    )

print(f"x_star = {x_star}")
print(f"Number of conditional modes after filtering: {len(mog_means)}")

## Data

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)
X = dist_utils.generate_mog_samples(25_000, mu_list, Sigma_list, alpha).float().to(device)
xh_cpu = X.detach().cpu().numpy()
plt.scatter(xh_cpu[:, 0], xh_cpu[:, 1], alpha=0.6, s=20)
plt.title("Scatter Plot of P(X,Y)")
plt.xlabel("X"); plt.ylabel("Y"); plt.grid(True); plt.show()

## Train Models
Same three models as the other experiment notebooks: Consistency Model and conditional/unconditional diffusion. Each cell loads an existing checkpoint if one is found (`load_checkpoint_with_hf_fallback`), otherwise trains it from scratch and saves a checkpoint — so this notebook is self-sufficient even with an empty `checkpoints/` dir.

### Consistency Model — $P(Y|X=x)$

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

B, C      = X.shape
nfeatures = C - CONDITION_ON
data_generator_cm = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha
)

Cos_ConsistencyModeliCT = ConsistencyModeliCT(
    nfeatures=nfeatures, condition_on=CONDITION_ON,
    nunits=NUNITS_CM, depth=NBLOCKS_CM
)

_loaded_cm = experiment_utils.load_checkpoint_with_hf_fallback(
    Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_cm:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    Cos_ConsistencyModeliCT.train_model(
        X=None, nepochs=NEPOCHS_CM, batch_size=BATCH_SIZE_CM,
        device=device, condition=CONDITION_ON,
        data_generator=data_generator_cm, use_improved_training=True
    )
    experiment_utils.save_model_checkpoint(
        Cos_ConsistencyModeliCT, "CM", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

### Diffusion — $P(Y|X=x)$
(This is the model whose `.sample()` calls are the target of `reuse_frac` for the plain LGD path.)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

X_train   = dist_utils.generate_mog_samples(1_000, mu_list, Sigma_list, alpha).float().to(device)
nfeatures = X_train.shape[1]

data_generator_diff_cond = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha, kernel_func=None
)

model_cond = Diffusion.DiffusionModel(
    nfeatures=nfeatures, nblocks=NBLOCKS, nunits=NUNITS,
    condition=True, condition_on=CONDITION_ON,
    diffusion_steps=DIFFUSION_STEPS
)

_loaded_diff_cond = experiment_utils.load_checkpoint_with_hf_fallback(
    model_cond, "Diffusion_cond", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_diff_cond:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    model_cond.train_model(
        None, data_generator=data_generator_diff_cond,
        nepochs=NEPOCHS, batch_size=BATCH_SIZE,
        condition_on=CONDITION_ON
    )
    experiment_utils.save_model_checkpoint(
        model_cond, "Diffusion_cond", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

### Diffusion — $P(X=x)$ (unconditional)

In [ ]:
experiment_utils.set_global_seed(GLOBAL_SEED)

data_generator_diff_uncond = partial(
    dist_utils.generate_mog_samples_not_differentiable,
    means=mu_list, variances=Sigma_list, weights=alpha,
    kernel_func=lambda X: X[:, :CONDITION_ON]
)

model_uncond = Diffusion.DiffusionModel(
    nfeatures=CONDITION_ON, nblocks=NBLOCKS, nunits=NUNITS,
    condition=False, diffusion_steps=DIFFUSION_STEPS
)

_loaded_diff_uncond = experiment_utils.load_checkpoint_with_hf_fallback(
    model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
    EXPERIMENT_NAME, GLOBAL_SEED, device
) if not FORCE_RETRAIN else False

if not _loaded_diff_uncond:
    experiment_utils.set_global_seed(GLOBAL_SEED)
    model_uncond.train_model(
        None, data_generator=data_generator_diff_uncond,
        nepochs=NEPOCHS, batch_size=BATCH_SIZE,
        condition_on=CONDITION_ON
    )
    experiment_utils.save_model_checkpoint(
        model_uncond, "Diffusion_uncond", CHECKPOINT_DIR,
        EXPERIMENT_NAME, GLOBAL_SEED
    )

## Run: `reuse_frac=0.0` vs `reuse_frac=0.5`, for LGD and LGD-CM

Same `N_ATTEMP_OPTIM` seeds for every setting (`experiment_utils.set_run_seed` reseeds identically per attempt, per `(method, reuse_frac)`), `return_history=True` so we get the per-step gradient norm, loss, and `n_reuse`/`n_new` back for each run. `run_sweep_point` takes the conditional model and `CM` flag as arguments so the same code drives both the plain-diffusion LGD path and the Consistency-Model LGD-F path.

In [ ]:
def run_sweep_point(cond_model, CM_flag, num_x_t, reuse_frac, label):
    x_t_list, final_loss_list, l2_gmm_list, l2_x_list, times, histories = [], [], [], [], [], []
    for i in trange(N_ATTEMP_OPTIM, desc=f"{label} | reuse_frac={reuse_frac}"):
        run_seed = experiment_utils.set_run_seed(GLOBAL_SEED, i)

        start_time = time.time()
        best_x_t, _, final_loss, history = Optimization.optimize_LGD(
            model_uncond, cond_model, mog_means, mog_variances, weights,
            mu_list, Sigma_list, alpha,
            nsamples=NSAMPLES_IN_OPTIM_FOR_MMD, loss="MMD", device=device,
            num_x_t=num_x_t, CM=CM_flag, reuse_frac=reuse_frac, return_history=True,
        )
        end_time = time.time()
        best_x_t = best_x_t.reshape(-1, 1)

        # Same metric as the other experiment notebooks: L2 distance between the
        # recovered condition x_pred and the true condition x_star, plus the
        # distribution-level L2 distance between the predicted and target
        # conditional GMMs at x_pred.
        x_pred_t = best_x_t.float().view(-1).cpu()
        mu_pred, Sigma_pred = dist_utils.compute_conditionals(mu_list, Sigma_list, x_pred_t)
        w_pred = dist_utils.compute_alpha(mu_list, Sigma_list, alpha, x_pred_t)
        l2_gmm = dist_utils.gmm_l2_distance(
            mu_pred, Sigma_pred, w_pred, mog_means, mog_variances, weights
        )
        l2_x = (x_pred_t - x_star.float().cpu()).pow(2).sum().sqrt().item()

        x_t_list.append(best_x_t)
        final_loss_list.append(final_loss.item())
        l2_gmm_list.append(l2_gmm)
        l2_x_list.append(l2_x)
        times.append(end_time - start_time)
        histories.append(history)
        print(f"[{label} | reuse_frac={reuse_frac} | {i+1}/{N_ATTEMP_OPTIM}] seed={run_seed} "
              f"| L2 GMM: {l2_gmm:.6f} | L2 to x*: {l2_x:.6f} "
              f"| final MMD: {final_loss.item():.6f} | time: {end_time - start_time:.2f}s")

    return {
        "x_t": x_t_list, "final_loss": final_loss_list,
        "l2_gmm": l2_gmm_list, "l2_x": l2_x_list,
        "times": times, "histories": histories,
    }

METHODS = {
    "LGD":    dict(cond_model=model_cond,             CM_flag=False, num_x_t=NUM_X_T_LGD),
    "LGD-CM": dict(cond_model=Cos_ConsistencyModeliCT, CM_flag=True,  num_x_t=NUM_X_T_LGD_CM),
}

results = {}   # results[method][reuse_frac] -> dict
for method, cfg in METHODS.items():
    results[method] = {}
    for rf in REUSE_FRACS:
        results[method][rf] = run_sweep_point(
            cfg["cond_model"], cfg["CM_flag"], cfg["num_x_t"], rf, label=method
        )

## Sanity check 1 — is `reuse_frac` actually reusing samples?
`n_reuse` should be 0 on the very first optimization step (no buffer yet) and then equal to `round(reuse_frac * NSAMPLES_IN_OPTIM_FOR_MMD)` on every step after that, for the `reuse_frac=0.5` runs — and 0 throughout for `reuse_frac=0.0`. Checked for both LGD and LGD-CM.

In [ ]:
for method in METHODS:
    for rf in REUSE_FRACS:
        h = results[method][rf]["histories"][0]   # first run's history
        n_reuse_after_first_step = [step["n_reuse"] for step in h[1:]]
        print(f"{method} | reuse_frac={rf}: step0 n_reuse={h[0]['n_reuse']}, "
              f"n_reuse thereafter (unique values)={sorted(set(n_reuse_after_first_step))}, "
              f"expected={round(rf * NSAMPLES_IN_OPTIM_FOR_MMD)}")

## Sanity check 2 — final L2-to-x* / L2-GMM & wall time

The headline metric here is `l2_gmm` (L2 distance between the predicted and target conditional GMMs at the recovered `x_pred`) and `l2_x` (L2 distance between `x_pred` and the true `x_star`) — the same metrics the other experiment notebooks report, not the internal MMD (`final_loss`) that `optimize_LGD` uses as its own convergence signal. `final_loss` is kept in the results/summary for reference but is no longer what's plotted.

In [ ]:
rows = []
for method in METHODS:
    for rf in REUSE_FRACS:
        r = results[method][rf]
        rows.append({
            "method":        method,
            "reuse_frac":    rf,
            "L2 GMM mean":   np.mean(r["l2_gmm"]),
            "L2 GMM std":    np.std(r["l2_gmm"]),
            "L2 to x* mean": np.mean(r["l2_x"]),
            "L2 to x* std":  np.std(r["l2_x"]),
            "MMD mean":      np.mean(r["final_loss"]),
            "MMD std":       np.std(r["final_loss"]),
            "Time mean (s)": np.mean(r["times"]),
            "Time std (s)":  np.std(r["times"]),
        })
summary_df = pd.DataFrame(rows)
summary_df

### Top 5 individual runs by final MMD loss

In [ ]:
rows_per_run = []
for method in METHODS:
    for rf in REUSE_FRACS:
        r = results[method][rf]
        for i in range(len(r["final_loss"])):
            rows_per_run.append({
                "method":      method,
                "reuse_frac":  rf,
                "run":         i,
                "seed":        GLOBAL_SEED + i,
                "MMD":         r["final_loss"][i],
                "L2 GMM":      r["l2_gmm"][i],
                "L2 to x*":    r["l2_x"][i],
                "Time (s)":    r["times"][i],
            })

top5_df = pd.DataFrame(rows_per_run).sort_values("MMD").head(5).reset_index(drop=True)
top5_df

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for row, method in enumerate(METHODS):
    axes[row, 0].boxplot([results[method][rf]["l2_gmm"] for rf in REUSE_FRACS],
                         labels=[str(rf) for rf in REUSE_FRACS])
    axes[row, 0].set_xlabel("reuse_frac"); axes[row, 0].set_ylabel("L2 GMM")
    axes[row, 0].set_title(f"{method}: L2(predicted GMM, target GMM)"); axes[row, 0].grid(True, alpha=0.3)

    axes[row, 1].boxplot([results[method][rf]["l2_x"] for rf in REUSE_FRACS],
                         labels=[str(rf) for rf in REUSE_FRACS])
    axes[row, 1].set_xlabel("reuse_frac"); axes[row, 1].set_ylabel("L2 to x*")
    axes[row, 1].set_title(f"{method}: L2(x_pred, x_star)"); axes[row, 1].grid(True, alpha=0.3)

    axes[row, 2].boxplot([results[method][rf]["times"] for rf in REUSE_FRACS],
                         labels=[str(rf) for rf in REUSE_FRACS])
    axes[row, 2].set_xlabel("reuse_frac"); axes[row, 2].set_ylabel("wall time (s)")
    axes[row, 2].set_title(f"{method}: per-run wall time"); axes[row, 2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Sanity check 3 — gradient-norm trajectory over the reverse-diffusion steps
This is the key mechanism check: with `reuse_frac=0.5`, gradient at each step flows only through the freshly-generated half of the MMD batch (the reused half is detached), so we expect a *noisier / not-necessarily-smaller* grad-norm curve with the same rough shape as `reuse_frac=0.0` — not a curve that's silently zero or identical to the baseline (which would mean the reuse path isn't wired in). Shown separately for LGD and LGD-CM.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, method in zip(axes, METHODS):
    for rf in REUSE_FRACS:
        histories = results[method][rf]["histories"]
        steps = [step["t"] for step in histories[0]]
        grad_matrix = np.array([[step["grad_norm"] for step in h] for h in histories])  # (n_runs, n_steps)
        mean_grad = grad_matrix.mean(axis=0)
        std_grad  = grad_matrix.std(axis=0)

        ax.plot(steps, mean_grad, label=f"reuse_frac={rf}")
        ax.fill_between(steps, mean_grad - std_grad, mean_grad + std_grad, alpha=0.2)

    ax.invert_xaxis()  # t goes from diffusion_steps-1 down to 1
    ax.set_xlabel("diffusion step t"); ax.set_ylabel("||grad x_t|| (mean ± std over runs)")
    ax.set_title(f"{method}: gradient-norm trajectory")
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Sanity check 4 — `reuse_frac=0.0` reproduces the pre-change code path
With `reuse_frac=0.0`, `n_reuse` is always 0, so every step generates the full `nsamples` batch fresh — identical to `optimize_LGD` before this change. This just re-affirms that from the collected history (all `n_new == NSAMPLES_IN_OPTIM_FOR_MMD`), for both methods.

In [ ]:
for method in METHODS:
    h0 = results[method][0.0]["histories"][0]
    all_full_batch = all(step["n_new"] == NSAMPLES_IN_OPTIM_FOR_MMD for step in h0)
    print(f"{method}: reuse_frac=0.0 always uses a full fresh batch every step: {all_full_batch}")

## Save results

In [ ]:
def to_python(val):
    if isinstance(val, torch.Tensor):
        return val.detach().cpu().tolist()
    if isinstance(val, np.ndarray):
        return val.tolist()
    if hasattr(val, "item"):
        return val.item()
    return val

out = {
    "experiment": EXPERIMENT_NAME,
    "seed": GLOBAL_SEED,
    "environment": env_info,
    "meta": {
        "n_attemp_optim": N_ATTEMP_OPTIM,
        "nsamples_in_optim_for_mmd": NSAMPLES_IN_OPTIM_FOR_MMD,
        "num_x_t_lgd": NUM_X_T_LGD,
        "num_x_t_lgd_cm": NUM_X_T_LGD_CM,
        "reuse_fracs": REUSE_FRACS,
    },
    "results": {
        method: {
            str(rf): {
                "final_loss": results[method][rf]["final_loss"],
                "l2_gmm": results[method][rf]["l2_gmm"],
                "l2_x": results[method][rf]["l2_x"],
                "times": results[method][rf]["times"],
                "histories": results[method][rf]["histories"],
            }
            for rf in REUSE_FRACS
        }
        for method in METHODS
    },
}

path = os.path.join(RESULTS_DIR, f"{EXPERIMENT_NAME}_hybrid_sanity_check_seed{GLOBAL_SEED}.json")
with open(path, "w") as f:
    json.dump(out, f, indent=2)
print(f"Results saved to {path}")